In [1]:
%reload_kedro

                    INFO     Resolved project path as:                                              ]8;id=474746;file://C:\Users\nicolas.betancourt\AppData\Local\anaconda3\envs\serpientes_env\lib\site-packages\kedro\ipython\__init__.py\__init__.py]8;;\:]8;id=280358;file://C:\Users\nicolas.betancourt\AppData\Local\anaconda3\envs\serpientes_env\lib\site-packages\kedro\ipython\__init__.py#175\175]8;;\
                             C:\Users\nicolas.betancourt\Documents\GitHub\pytorch\serpientes-de-col                
                             ombia.                                                                                
                             To set a different path, run '%reload_kedro <project_root>'                           

[03/20/26 11:41:39] INFO     Kedro is sending anonymous usage data with the sole purpose of improving ]8;id=386013;file://C:\Users\nicolas.betancourt\AppData\Local\anaconda3\envs\serpientes_env\lib\site-packages\kedro_telemetry\plugin.py\plugin.py]8;;\:]8;id=421069;file://C:\Users\nicolas.betancourt\AppData\Local\anaconda3\envs\serpientes_env\lib\site-packages\kedro_telemetry\plugin.py#243\243]8;;\
                             the product. No personal data or IP addresses are stored on our side. To              
                             opt out, set the `KEDRO_DISABLE_TELEMETRY` or `DO_NOT_TRACK` environment              
                             variables, or create a `.telemetry` file in the current working                       
                             directory with the contents `consent: false`. To hide this message,                   
                             explicitly grant or deny consent. Read more at                                        
                             https://docs.kedro.org/en/stable/configuration/telemetry.html                         

[03/20/26 11:41:40] INFO     Kedro project serpientes_de_colombia                                   ]8;id=416186;file://C:\Users\nicolas.betancourt\AppData\Local\anaconda3\envs\serpientes_env\lib\site-packages\kedro\ipython\__init__.py\__init__.py]8;;\:]8;id=876254;file://C:\Users\nicolas.betancourt\AppData\Local\anaconda3\envs\serpientes_env\lib\site-packages\kedro\ipython\__init__.py#141\141]8;;\

                    INFO     Defined global variable 'context', 'session', 'catalog' and            ]8;id=293787;file://C:\Users\nicolas.betancourt\AppData\Local\anaconda3\envs\serpientes_env\lib\site-packages\kedro\ipython\__init__.py\__init__.py]8;;\:]8;id=943169;file://C:\Users\nicolas.betancourt\AppData\Local\anaconda3\envs\serpientes_env\lib\site-packages\kedro\ipython\__init__.py#142\142]8;;\
                             'pipelines'                                                                           

                    INFO     Registered line magic 'run_viz'                                        ]8;id=263143;file://C:\Users\nicolas.betancourt\AppData\Local\anaconda3\envs\serpientes_env\lib\site-packages\kedro\ipython\__init__.py\__init__.py]8;;\:]8;id=872399;file://C:\Users\nicolas.betancourt\AppData\Local\anaconda3\envs\serpientes_env\lib\site-packages\kedro\ipython\__init__.py#148\148]8;;\

In [2]:
import numpy as np
import pandas as pd
from torch.utils.data import random_split


In [3]:
image_urls   =catalog.load('image_urls@pandas')
train_validation_size   =catalog.load('params:train_validation_size') 
validation_size=catalog.load('params:validation_size')

                    INFO     Loading data from image_urls@pandas (CSVDataset)...                ]8;id=575172;file://C:\Users\nicolas.betancourt\AppData\Local\anaconda3\envs\serpientes_env\lib\site-packages\kedro\io\data_catalog.py\data_catalog.py]8;;\:]8;id=148477;file://C:\Users\nicolas.betancourt\AppData\Local\anaconda3\envs\serpientes_env\lib\site-packages\kedro\io\data_catalog.py#539\539]8;;\

                    INFO     Loading data from params:train_validation_size (MemoryDataset)...  ]8;id=607998;file://C:\Users\nicolas.betancourt\AppData\Local\anaconda3\envs\serpientes_env\lib\site-packages\kedro\io\data_catalog.py\data_catalog.py]8;;\:]8;id=18287;file://C:\Users\nicolas.betancourt\AppData\Local\anaconda3\envs\serpientes_env\lib\site-packages\kedro\io\data_catalog.py#539\539]8;;\

                    INFO     Loading data from params:validation_size (MemoryDataset)...        ]8;id=156788;file://C:\Users\nicolas.betancourt\AppData\Local\anaconda3\envs\serpientes_env\lib\site-packages\kedro\io\data_catalog.py\data_catalog.py]8;;\:]8;id=848227;file://C:\Users\nicolas.betancourt\AppData\Local\anaconda3\envs\serpientes_env\lib\site-packages\kedro\io\data_catalog.py#539\539]8;;\

In [4]:
image_urls=image_urls[image_urls['label']!='macabrel']

In [ ]:
min_train_fraction=0.8*image_urls['label'].value_counts().min()

train_validation=image_urls[image_urls.assign(random=np.random.normal(0,1)).groupby('label')['random'].transform('rank','first')<=min_train_fraction]
test=image_urls[~image_urls['observation_id'].isin(train_validation['observation_id'])]

train_validation['set']=train_validation.groupby('label')['label'].transform(lambda x: np.random.binomial(n=1,p=validation_size, size=len(x)))

train, validation=train_validation[train_validation['set']==0], train_validation[train_validation['set']==1]

In [15]:
image_urls['label'].value_counts()



label
falsa_vibora    2517
falsa_coral     1625
vibora          1078
coral           1049
Name: count, dtype: int64

In [14]:
train_validation['label'].value_counts()


label
vibora          839
coral           839
falsa_vibora    839
falsa_coral     839
Name: count, dtype: int64

In [17]:
test['label'].value_counts()



label
falsa_vibora    1678
falsa_coral      786
vibora           238
coral            209
Name: count, dtype: int64

In [18]:
train['label'].value_counts()



label
falsa_coral     762
falsa_vibora    761
coral           757
vibora          756
Name: count, dtype: int64

In [19]:
validation['label'].value_counts()



label
vibora          83
coral           82
falsa_vibora    78
falsa_coral     77
Name: count, dtype: int64